# Player Value Forecaster: end to end

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jackstev-edu/player-value-forecaster/blob/main/notebooks/02_end_to_end.ipynb)

CMU 24-679 Project 1, Yunus Polatoglu and Jack Stevens. From the published dataset to
forecasts, in one run on a free CPU runtime (about 5 minutes; **Runtime → Run all**):

1. Install the project code from GitHub and load the [dataset](https://huggingface.co/datasets/ypolatog/football-contracts-500).
2. **Train and cross-validate** the from-scratch model (LightGBM quantiles) against three baselines.
3. Run the **off-the-shelf** model (Chronos-Bolt, zero-shot) on the same rows and compare.
4. Load the **published** model from the Hub and forecast a player you describe.

Model cards: [LightGBM core](https://huggingface.co/ypolatog/player-value-lgbm-core) ·
[Chronos-Bolt](https://huggingface.co/ypolatog/player-value-chronos-bolt) ·
App: [Space](https://huggingface.co/spaces/ypolatog/player-value-forecaster)

In [ ]:
# 1. Code. Colab already has pandas, numpy, scikit-learn, lightgbm, torch and huggingface_hub.
import os, subprocess, sys
REPO_REF = "package/prototype"   # branch or tag of the project repo; "main" once merged
if not os.path.exists("player-value-forecaster"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF,
                    "https://github.com/jackstev-edu/player-value-forecaster.git"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "chronos-forecasting", "pyyaml",
                "-e", "player-value-forecaster"], check=True)
sys.path.insert(0, os.path.abspath("player-value-forecaster/src"))

In [ ]:
import os
import pandas as pd

# Reads the published Hugging Face dataset. PVF_LOCAL_DATASET points at a local copy instead
# (used to test the notebook before publishing).
DATASET = "ypolatog/football-contracts-500"
BASE = os.environ.get("PVF_LOCAL_DATASET", f"hf://datasets/{DATASET}")

def read(path):
    full = f"{BASE}/{path}"
    return pd.read_csv(full) if path.endswith(".csv") else pd.read_parquet(full)

manual = read("samples/manual.parquet")
augmented = read("samples/augmented.parquet")
contracts = read("contracts/manual.csv")
history = read("value_history.parquet")
print(f"manual {len(manual)} rows, augmented {len(augmented)} rows, "
      f"contracts {len(contracts)}, value history {len(history)} valuations")

## 2. The from-scratch model under player-grouped cross-validation

Five folds; every player sits on one side of each fold. Augmented rows train only; **only the
500 hand-collected rows are scored**. Baselines: no change, an age × position median curve, and
ridge regression. The model: LightGBM at quantiles 0.1/0.5/0.9 on four features, early-stopped
on held-out players, with the 10–90% band conformally calibrated.

In [ ]:
from pvf.config import load_config
from pvf.evaluation.cv import cross_validate, summarize, paired_diff
from pvf.features.samples import CORE_FEATURES

cfg = load_config("player-value-forecaster/configs/config.yaml")
m, seed = cfg["model"], cfg["project"]["seed"]
samples = pd.concat([manual, augmented], ignore_index=True)
samples = samples[samples["y_h1"].notna()]
oof = cross_validate(samples, {"gbm_core": CORE_FEATURES}, quantiles=m["quantiles"], params=m["gbm"],
                     n_splits=cfg["split"]["n_splits"], seed=seed, val_share=m["val_share"],
                     min_count=m["baseline_min_count"])
table = summarize(oof, samples)
table[table["subset"] == "all"].round(3)

`mae_log` 0.61 is a typical miss of about ×1.8; `coverage_80` should be near 0.80. The ridge
baseline ties LightGBM on the median; LightGBM earns its place through the calibrated range.

## 3. The off-the-shelf model on the same rows

Chronos-Bolt reads only each player's monthly value history **up to the anchor** and forecasts
12 months ahead, with no training. Scored on the same folds' manual rows.

In [ ]:
import numpy as np
from pvf.evaluation.splits import player_folds
from pvf.models.foundation import ChronosForecaster, contexts_for

history["date"] = pd.to_datetime(history["date"])
forecaster = ChronosForecaster("amazon/chronos-bolt-small")
contexts = contexts_for(history, samples["player_id"], pd.to_datetime(samples["anchor_date"]))
chronos = forecaster.predict(contexts, samples["value_now"].to_numpy())
chronos.index = samples.index

rows = []
for k, (_, test_idx) in enumerate(player_folds(samples, cfg["split"]["n_splits"], seed)):
    rows.append(pd.DataFrame({"fold": k, "row": test_idx, "model": "chronos_zero_shot",
                              "pred": chronos.loc[test_idx, "q50"].to_numpy(),
                              "q10": chronos.loc[test_idx, "q10"].to_numpy(),
                              "q90": chronos.loc[test_idx, "q90"].to_numpy()}))
both = pd.concat([oof, *rows], ignore_index=True)
compare = summarize(both, samples)
compare = compare[compare["subset"] == "all"].drop(columns="subset").round(3)
compare

In [ ]:
diffs = paired_diff(both, samples, reference="gbm_core",
                    others=["no_change", "age_position", "linear", "chronos_zero_shot"])
print("log MAE minus LightGBM (positive = worse), 95% bootstrap interval")
diffs.round(3)

**Reading it.** On these future stars LightGBM wins clearly, and Chronos does worse than "no
change": 79% of the players rose, often several-fold, and a history-only forecaster expects a
series to settle back. Chronos' coverage shows as blank because 14 players with no history get
no band. The model cards add the opposite check on 2,000 ordinary league players: there
Chronos predicts a rise for 28% (the truth was 28%) and LightGBM for 97%. Each model leans
towards the population it learned from: LightGBM towards our sample, Chronos towards time
series in general.

## 4. The published model: forecast a player

The app's exact model, downloaded from the Hub. Change the inputs and re-run the cell.

In [ ]:
from huggingface_hub import snapshot_download
from pvf.models.gbm import QuantileGBM

local = os.environ.get("PVF_LOCAL_MODEL")
model = QuantileGBM.load(local or snapshot_download("ypolatog/player-value-lgbm-core"))

age = 20.5                 #@param {type:"number"}
position = "Attack"        #@param ["Attack", "Midfield", "Defender", "Goalkeeper"]
value_now_eur = 25_000_000 #@param {type:"number"}
value_a_year_ago_eur = 15_000_000 #@param {type:"number"}

X = pd.DataFrame({"age": [age], "position": [position], "value_now": [value_now_eur],
                   "value_change_12m": [np.log(value_now_eur / value_a_year_ago_eur)]})
q = model.predict(X).iloc[0]
lo, mid, hi = (value_now_eur * np.exp(q[c]) / 1e6 for c in ["q10", "q50", "q90"])
print(f"Value in one season: EUR {mid:.1f}M (likely range {lo:.1f}M to {hi:.1f}M; "
      f"today {value_now_eur / 1e6:.1f}M)")
print("Remember the bias: trained on future stars, so read the direction with care for ordinary players.")

## 5. Both models on one real player

In [ ]:
import matplotlib.pyplot as plt

name = "Kylian Mbappé"   #@param {type:"string"}
pick = manual[manual["player_name"] == name]
row = pick.iloc[0]
anchor = pd.Timestamp(row["anchor_date"])
h = history[history["player_id"] == row["player_id"]]
g = model.predict(pick).iloc[0]
c = forecaster.predict(contexts_for(history, [row["player_id"]], [anchor]), np.array([row["value_now"]])).iloc[0]
target = anchor + pd.DateOffset(years=1)
fig, ax = plt.subplots(figsize=(9, 4))
ax.step(h["date"], h["market_value_in_eur"] / 1e6, where="post", color="#52514e", lw=2, label="Transfermarkt value")
for off, f, col, lab in [(-20, g, "#2a78d6", "LightGBM (from scratch)"), (20, c, "#eb6834", "Chronos (off-the-shelf)")]:
    x = target + pd.Timedelta(days=off)
    v = row["value_now"] * np.exp([f["q10"], f["q50"], f["q90"]]) / 1e6
    ax.errorbar([x], [v[1]], yerr=[[v[1] - v[0]], [v[2] - v[1]]], fmt="o", color=col, capsize=4, lw=2, label=lab)
ax.axvline(anchor, color="grey", ls="--", lw=1)
ax.set(title=f"{name}: forecasts made on {anchor.date()}, one season ahead (10–90% range)", ylabel="EUR million")
ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False); plt.show()
print(f"Actual change: {row['y_h1']:+.2f} log | LightGBM {g['q50']:+.2f} | Chronos {c['q50']:+.2f}")

Note: this player's own row was in the published model's training data, so this plot
illustrates the outputs. The honest scores are the cross-validated ones in sections 2 and 3.